# DONKI Quickstart — the May 2024 Gannon storm

This notebook demonstrates the `DonkiAdapter` against the
May 2024 Gannon G5 storm window (2024-05-08 → 2024-05-15).
We fetch CMEs and solar flares, then walk DONKI's *intelligent
linkages* to show how a geomagnetic storm traces back to its
originating coronal mass ejections.

The Gannon storm is the strongest event in HELIOS's pre-registered hold-out set (proposal §3 Table 3-1) and the
marquee example for the precision-ag GNSS slice (§1.3).


In [ ]:
from datetime import UTC, datetime

from helios_connectors import DonkiAdapter
from helios_connectors.adapters.donki import DONKI_KAUAI_BASE_URL

## 1. Fetch CMEs and flares for the Gannon window

We point the adapter at CCMC's `kauai` endpoint to avoid the
`api.nasa.gov` DEMO_KEY rate cap. Set `NASA_API_KEY` in your
environment to use the default `api.nasa.gov` route instead.


In [ ]:
start = datetime(2024, 5, 8, tzinfo=UTC)
end = datetime(2024, 5, 15, tzinfo=UTC)

async with DonkiAdapter(base_url=DONKI_KAUAI_BASE_URL) as donki:
    cmes = [r async for r in donki.fetch_cme(start=start, end=end)]
    flares = [r async for r in donki.fetch_flr(start=start, end=end)]
    gsts = [r async for r in donki.fetch_gst(start=start, end=end)]

print(f"CMEs:   {len(cmes)}")
print(f"Flares: {len(flares)}")
print(f"GSTs:   {len(gsts)}")

## 2. Inspect a normalized record

Each record carries a science payload (`value`) plus a
`ProvenanceRecord` describing where the value came from
and what upstream events it depends on.


In [ ]:
cme = cmes[0]
print("record_type:", cme.record_type)
print("event_time:", cme.event_time)
print("provenance.id:", cme.provenance.id)
print("provenance.model_id:", cme.provenance.model_id)
print("provenance.lineage:", cme.provenance.lineage)
print("value keys:", sorted(cme.value.keys())[:8])

## 3. Trace the Gannon G5 storm back to its source CMEs

DONKI's geomagnetic-storm records carry `linkedEvents` pointing
back at the originating coronal mass ejections. The HELIOS
adapter surfaces these as `provenance.lineage`. This is the
key affordance for downstream fusion: every output traces to
every contributing upstream event.


In [ ]:
gannon = next(g for g in gsts if g.provenance.id.startswith("2024-05-10"))
print(f"Gannon GST id: {gannon.provenance.id}")
print(f"Event time: {gannon.event_time}")
print()
print(f"Lineage ({len(gannon.provenance.lineage)} upstream events):")
for upstream in gannon.provenance.lineage:
    print(f"  - {upstream}")

## 4. Plot a timeline of events

A simple scatterplot of event_time per record_type makes the
cadence of the storm visible at a glance: a burst of flares,
the CMEs that propagated outward, and the resulting GST.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
for label, recs, color in [
    ("Flares", flares, "tab:red"),
    ("CMEs", cmes, "tab:blue"),
    ("GSTs", gsts, "tab:purple"),
]:
    times = [r.event_time for r in recs]
    ax.scatter(times, [label] * len(times), color=color, s=40, alpha=0.7)
ax.set_title("DONKI events: May 2024 Gannon storm window")
ax.set_xlabel("UTC")
ax.grid(alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## What's next

- Swap `DonkiAdapter` for `SwpcAdapter` (forthcoming) to ingest
  the operational `Kp` series alongside DONKI events.
- Drop the records into `helios-fusion-engine` to feed a BMA
  fusion pipeline (proposal §2 Obj. 2).
- Each `ProvenanceRecord` is forward-compatible with the
  `helios-provenance-spec` v0.1 schema; once that ships, this
  notebook will validate every record against the schema.
